In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import ndimage
from skimage.filters import threshold_otsu
import pandas as pd

In [ ]:
INPUT_PATH = Path(
    "../data/sample/processed/stage2_preprocessing/44b6_0113de3b_t00_z20_corrected.npy"
)

corrected = np.load(INPUT_PATH)

print("Loaded:", INPUT_PATH)
print("Shape:", corrected.shape)
print("Dtype:", corrected.dtype)
print("Min:", corrected.min())
print("Max:", corrected.max())

Otsu thresholding

In [ ]:
threshold = threshold_otsu(corrected)

binary_mask = corrected > threshold

print("Threshold:", threshold)
print("Mask shape:", binary_mask.shape)

In [ ]:
z = 32

fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 5)
)

# Preprocessed image
axes[0].imshow(
    corrected[z],
    cmap="gray"
)

axes[0].set_title(
    f"Preprocessed Image (Z={z})"
)

# Otsu segmentation
axes[1].imshow(
    binary_mask[z],
    cmap="gray"
)

axes[1].set_title(
    f"Otsu Segmentation (Z={z})"
)

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
print("Binary mask shape:", binary_mask.shape)
print("Binary mask dtype:", binary_mask.dtype)

# ---------------------------------------------------------
# 1. Connected component labeling
# ---------------------------------------------------------

# 3D connectivity structure
structure = ndimage.generate_binary_structure(
    rank=3,
    connectivity=2
)

labeled, num_objects = ndimage.label(
    binary_mask,
    structure=structure
)

print("Number of connected components:", num_objects)

In [ ]:
# ---------------------------------------------------------
# 2. Calculate size of each connected component
# ---------------------------------------------------------

object_sizes = np.bincount(
    labeled.ravel()
)

# Object 0 is the background, so remove it
object_sizes = object_sizes[1:]

print("Number of objects:", len(object_sizes))
print("Smallest object:", object_sizes.min())
print("Largest object:", object_sizes.max())
print("Median object size:", np.median(object_sizes))

In [ ]:
plt.figure(figsize=(10, 5))

plt.hist(
    object_sizes,
    bins=100
)

plt.xlabel("Object size (voxels)")
plt.ylabel("Number of objects")
plt.title("Connected Component Size Distribution")

plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

plt.hist(
    object_sizes,
    bins=50
)

plt.xscale("log")

plt.xlabel("Object size (voxels)")
plt.ylabel("Number of objects")

plt.title("3D Segmented Object Size Distribution")

plt.show()

In [ ]:
# ---------------------------------------------------------
# 3. Visualize connected components
# ---------------------------------------------------------

z = 32

fig, axes = plt.subplots(
    1,
    3,
    figsize=(18, 6)
)

# Preprocessed image
axes[0].imshow(
    corrected[z],
    cmap="gray"
)

axes[0].set_title(
    f"Preprocessed Image (Z={z})"
)

# Binary mask
axes[1].imshow(
    binary_mask[z],
    cmap="gray"
)

axes[1].set_title(
    f"Otsu Binary Mask (Z={z})"
)

# Connected components
axes[2].imshow(
    labeled[z],
    cmap="nipy_spectral"
)

axes[2].set_title(
    f"Connected Components (Z={z})"
)

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# --------------------------------------------------
# Label connected components
# --------------------------------------------------

labeled, num_objects = ndimage.label(binary_mask)

print("Number of objects:", num_objects)


# --------------------------------------------------
# Calculate object properties
# --------------------------------------------------

objects = ndimage.find_objects(labeled)

object_data = []

for object_id, obj_slice in enumerate(objects, start=1):

    if obj_slice is None:
        continue

    # Extract voxel coordinates
    coords = np.where(labeled[obj_slice] == object_id)

    # Convert local coordinates to global coordinates
    z_coords = coords[0] + obj_slice[0].start
    y_coords = coords[1] + obj_slice[1].start
    x_coords = coords[2] + obj_slice[2].start

    volume = len(z_coords)

    centroid_z = np.mean(z_coords)
    centroid_y = np.mean(y_coords)
    centroid_x = np.mean(x_coords)

    object_data.append({
        "object_id": object_id,
        "volume": volume,
        "centroid_z": centroid_z,
        "centroid_y": centroid_y,
        "centroid_x": centroid_x
    })


# --------------------------------------------------
# Create DataFrame
# --------------------------------------------------

objects_df = pd.DataFrame(object_data)

print(objects_df.head())

print("\nObject statistics:")
print(objects_df["volume"].describe())

In [ ]:
print("Smallest objects:")
display(
    objects_df
    .sort_values("volume")
    .head(20)
)

In [ ]:
import matplotlib.pyplot as plt

fig = plt.figure(figsize=(10, 8))

ax = fig.add_subplot(111, projection="3d")

ax.scatter(
    objects_df["centroid_x"],
    objects_df["centroid_y"],
    objects_df["centroid_z"],
    s=20
)

ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z")

ax.set_title("3D Centroids of Detected Objects")

plt.show()

Some cells are at the boundary of the space. The smaller cells could be those located at the edge of the volume. So we are going to further analyze those boundary objects

In [ ]:
object_data = []

for object_id, obj_slice in enumerate(objects, start=1):

    if obj_slice is None:
        continue

    # Coordinates of this component
    coords = np.where(labeled[obj_slice] == object_id)

    z_coords = coords[0] + obj_slice[0].start
    y_coords = coords[1] + obj_slice[1].start
    x_coords = coords[2] + obj_slice[2].start

    volume = len(z_coords)

    centroid_z = np.mean(z_coords)
    centroid_y = np.mean(y_coords)
    centroid_x = np.mean(x_coords)

    # Bounding box
    z_min = z_coords.min()
    z_max = z_coords.max()

    y_min = y_coords.min()
    y_max = y_coords.max()

    x_min = x_coords.min()
    x_max = x_coords.max()

    # Check whether object touches volume boundaries
    touches_boundary = (
            z_min == 0 or
            z_max == binary_mask.shape[0] - 1 or
            y_min == 0 or
            y_max == binary_mask.shape[1] - 1 or
            x_min == 0 or
            x_max == binary_mask.shape[2] - 1
    )

    object_data.append({
        "object_id": object_id,

        "volume": volume,

        "centroid_z": centroid_z,
        "centroid_y": centroid_y,
        "centroid_x": centroid_x,

        "z_min": z_min,
        "z_max": z_max,

        "y_min": y_min,
        "y_max": y_max,

        "x_min": x_min,
        "x_max": x_max,

        "touches_boundary": touches_boundary
    })


objects_df = pd.DataFrame(object_data)

print(objects_df.head())

In [ ]:
print(
    objects_df["touches_boundary"].value_counts()
)

In [ ]:
small_objects = objects_df[
    objects_df["volume"] < 200
    ].sort_values("volume")

display(
    small_objects[
        [
            "object_id",
            "volume",
            "centroid_z",
            "centroid_y",
            "centroid_x",
            "touches_boundary"
        ]
    ]
)

Save results

In [ ]:
from pathlib import Path

# Project root
PROJECT_ROOT = Path.cwd()

# If running from notebooks/, adjust this if necessary
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Dataset name
SAMPLE_NAME = "44b6_0113de3b"

# Stage 3 output directory
STAGE3_DIR = (
        PROJECT_ROOT
        / "data"
        / "sample"
        / "processed"
        / "stage3_segmentation"
        / SAMPLE_NAME
)

STAGE3_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Stage 3 output directory:")
print(STAGE3_DIR)

In [ ]:
np.save(
    STAGE3_DIR / "binary_mask.npy",
    binary_mask
)

print(
    "Saved binary mask:",
    STAGE3_DIR / "binary_mask.npy"
)

In [ ]:
np.save(
    STAGE3_DIR / "labeled.npy",
    labeled
)

print(
    "Saved labeled segmentation:",
    STAGE3_DIR / "labeled.npy"
)

In [ ]:
objects_df.to_csv(
    STAGE3_DIR / "objects.csv",
    index=False
)

print(
    "Saved object table:",
    STAGE3_DIR / "objects.csv"
)

In [ ]:
import json

metadata = {
    "sample_name": SAMPLE_NAME,

    "volume_shape": list(binary_mask.shape),

    "num_objects": int(num_objects),

    "segmentation_method": "Otsu",

    "binary_mask_dtype": str(binary_mask.dtype),

    "labeled_dtype": str(labeled.dtype),

    "object_table": "objects.csv",

    "binary_mask": "binary_mask.npy",

    "labeled_segmentation": "labeled.npy"
}

with open(
        STAGE3_DIR / "metadata.json",
        "w"
) as f:

    json.dump(
        metadata,
        f,
        indent=4
    )

print(
    "Saved metadata:",
    STAGE3_DIR / "metadata.json"
)

In [ ]:
print("\n" + "=" * 60)
print("STAGE 3 OUTPUTS")
print("=" * 60)

for file in STAGE3_DIR.iterdir():
    print(
        file.name,
        "→",
        f"{file.stat().st_size / (1024**2):.2f} MB"
    )

Evaluate against ground truth

In [ ]:
import numpy as np
import pandas as pd


def evaluate_stage3_against_ground_truth(
        binary_mask,
        ground_truth_nodes,
        timepoint=0
):
    """
    Evaluate Stage 3 semantic segmentation against ground-truth
    cell centers for a single timepoint.

    Parameters
    ----------
    binary_mask : np.ndarray
        3D binary segmentation mask with shape (Z, Y, X).

    ground_truth_nodes : pd.DataFrame
        Ground-truth node table containing:
        node_id, t, z, y, x

    timepoint : int
        Timepoint to evaluate.

    Returns
    -------
    evaluation_df : pd.DataFrame
        Ground-truth cells with Stage 3 detection status.

    summary : dict
        Summary statistics.
    """

    # ---------------------------------------------------------
    # 1. Select ground-truth cells for this timepoint
    # ---------------------------------------------------------

    gt_t = ground_truth_nodes[
        ground_truth_nodes["t"] == timepoint
        ].copy()

    gt_t = gt_t.reset_index(drop=True)

    print("=" * 60)
    print(f"STAGE 3 EVALUATION — TIMEPOINT {timepoint}")
    print("=" * 60)

    print(
        f"Ground-truth cells at t={timepoint}: "
        f"{len(gt_t)}"
    )

    # ---------------------------------------------------------
    # 2. Check each ground-truth center against binary mask
    # ---------------------------------------------------------

    detected = []

    for _, row in gt_t.iterrows():

        z = int(row["z"])
        y = int(row["y"])
        x = int(row["x"])

        # Make sure coordinate is inside mask
        inside_volume = (
                0 <= z < binary_mask.shape[0]
                and
                0 <= y < binary_mask.shape[1]
                and
                0 <= x < binary_mask.shape[2]
        )

        if inside_volume:
            mask_value = binary_mask[z, y, x]
        else:
            mask_value = 0

        detected.append(
            mask_value > 0
        )

    gt_t["stage3_detected"] = detected

    # ---------------------------------------------------------
    # 3. Calculate statistics
    # ---------------------------------------------------------

    total_cells = len(gt_t)

    detected_cells = int(
        gt_t["stage3_detected"].sum()
    )

    missed_cells = (
            total_cells - detected_cells
    )

    detection_rate = (
        detected_cells / total_cells
        if total_cells > 0
        else 0.0
    )

    # ---------------------------------------------------------
    # 4. Print results
    # ---------------------------------------------------------

    print()
    print("RESULTS")
    print("-" * 60)

    print(
        f"Ground-truth cells : {total_cells}"
    )

    print(
        f"Detected by Stage 3: {detected_cells}"
    )

    print(
        f"Missed by Stage 3  : {missed_cells}"
    )

    print(
        f"Detection rate     : "
        f"{detection_rate * 100:.2f}%"
    )

    # ---------------------------------------------------------
    # 5. Summary
    # ---------------------------------------------------------

    summary = {
        "timepoint": timepoint,
        "ground_truth_cells": total_cells,
        "detected_cells": detected_cells,
        "missed_cells": missed_cells,
        "detection_rate": detection_rate,
    }

    return gt_t, summary

In [ ]:
GROUND_TRUTH_NODES = (
        PROJECT_ROOT
        / "data"
        / "sample"
        / "biohub_5samples_20timepoints"
        / "train"
        / SAMPLE_NAME
        / "ground_truth"
        / "ground_truth_nodes.csv"
)

ground_truth_nodes = pd.read_csv(
    GROUND_TRUTH_NODES
)

print(
    ground_truth_nodes.head()
)

In [ ]:
gt_t0, summary_t0 = evaluate_stage3_against_ground_truth(
    binary_mask=binary_mask,
    ground_truth_nodes=ground_truth_nodes,
    timepoint=0
)

In [ ]:
print(
    ground_truth_nodes.groupby("t").size()
)
print(
    "Total ground-truth nodes:",
    len(ground_truth_nodes)
)

print(
    "Unique timepoints:",
    ground_truth_nodes["t"].nunique()
)

So the ground truth is not a complete cell segmentation annotation for the volume. It appears to represent a single tracked cell lineage across three timepoints: